# Classificação de Materiais Recicláveis utilizando ResNet-50 (Parte 1)
*ESZA019 – Visao Computacional - 2026.2*

Integrantes - Grupo 2:
- Cesar de Jesus Carvalho 
- Mariana Chiara Travassos Sarinho
- Vinícius de Marchi Costa

Este notebook tem como objetivo carregar e pré-processar o *dataset **TrashNet***, assim como desenvolver e treinar um modelo de aprendizado profundo baseado na arquitetura ResNet-50 para classificar imagens de materiais recicláveis.

O **fluxo** desenvolvido nesse notebook compreende:
1. Configuração do ambiente
2. Leitura do conjunto de dados
3. Análise exploratória das classes
4. Separação dos dados em treinamento, validação e teste
5. Pré-processamento das imagens
6. Aplicação de Data Augmentation
7. Construção da ResNet-50 utilizando Transfer Learning
8. Treinamento inicial do modelo
9. Fine-tuning
10. Avaliação do modelo e geração de métricas

Ao final, o modelo treinado será salvo para ser utilizado posteriormente no notebook `02.webcam.ipynb`, responsável pela classificação de materiais recicláveis em tempo real utilizando uma webcam.

------------------
## Configuração do ambiente

As principais bibliotecas utilizadas são:
- **TensorFlow/Keras:** construção e treinamento da rede neural;
- **NumPy:** manipulação de arrays e operações numéricas;
- **Pandas:** organização dos resultados;
- **Matplotlib:** visualização das imagens e curvas de treinamento;
- **Scikit-learn:** cálculo das métricas de avaliação e matriz de confusão

In [ ]:
# Instala todas as bibliotecas necessárias
python -m pip install -r requirements.txt

In [ ]:
import os
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

As imagens serão redimensionadas para **224 × 224 pixels**, tamanho tradicionalmente utilizado como entrada para a ResNet-50 pré-treinada no conjunto de dados *ImageNet*.

Também serão definidos:
- **tamanho do batch** - quantidade de imagens processadas simultaneamente;
- **proporção destinada à validação** - fração do *dataset* destinada a avaliar o desempenho;
- **número de épocas** - número de vezes que o conjunto de treinamento será apresentado ao modelo;
- **taxa de aprendizado (*learning rate*)** - controla o tamanho dos ajustes nos pesos;
- **semente aleatória (*random seed*)** - garante a reprodutibilidade dos experimentos;
- **diretórios para armazenamento dos resultados** - caminhos específicos para salvar artefatos.

In [ ]:
# ==========================================================
# PARÂMETROS
# ==========================================================

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_SPLIT = 0.70
VALIDATION_SPLIT = 0.15
TEST_SPLIT = 0.15
EPOCHS_INITIAL = 10
EPOCHS_FINE_TUNING = 5
INITIAL_LEARNING_RATE = 1e-4
FINE_TUNING_LEARNING_RATE = 1e-5
DROPOUT_RATE = 0.40
FINE_TUNE_AT = 140

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Configurações carregadas.")

Recomenda-se, para este notebook, utilizar uma estrutura semelhante à:

```text
    Projeto - Visão Computacional/
    │
    ├── 01_modelo.ipynb
    ├── 02_webcam.ipynb
    │
    ├── dataset/
    │   └── images/
    │       ├── cardboard/
    │       ├── glass/
    │       ├── metal/
    │       ├── paper/
    │       ├── plastic/
    │       └── trash/
    │
    ├── models/
    │
    └── results/

In [ ]:
# ==========================================================
# PRINCIPAIS CAMINHOS E DIRETÓRIOS
# ==========================================================

# Caminho principal do dataset
DATASET_PATH = "./dataset"

# Diretórios de saída
MODELS_DIR = "./models"
RESULTS_DIR = "./results"

# Criação dos diretórios
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Dataset:", DATASET_PATH)
print("Modelos:", MODELS_DIR)
print("Resultados:", RESULTS_DIR)

A ResNet-50 possui milhões de parâmetros e o treinamento pode ser consideravelmente acelerado utilizando uma GPU, logo, será verificado se o TensorFlow reconhece uma GPU.

In [ ]:
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("\nGPU(s) encontrada(s):")
    for gpu in gpus:
        print(" -", gpu)
else:
    print("\nNenhuma GPU encontrada.")
    print("O treinamento será realizado utilizando a CPU.")

------------
## Dataset

### Verificação do dataset

O *dataset* deve estar organizado por diretórios, em que cada diretório representa uma classe.

In [ ]:
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset não encontrado em: {DATASET_PATH}"
    )

class_names = sorted([
    folder for folder in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, folder))
])

print("Classes encontradas:")
for i, class_name in enumerate(class_names):
    print(f"{i}: {class_name}")

NUM_CLASSES = len(class_names)

print("\nNúmero total de classes:", NUM_CLASSES)

In [ ]:
# ==========================================================
# CONTAGEM DAS IMAGENS
# ==========================================================

image_extensions = (
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

class_counts = {}

for class_name in class_names:

    class_path = os.path.join(
        DATASET_PATH,
        class_name
    )

    count = sum(
        1
        for file in os.listdir(class_path)
        if file.lower().endswith(image_extensions)
    )

    class_counts[class_name] = count

class_counts_df = pd.DataFrame(
    list(class_counts.items()),
    columns=["Classe", "Quantidade"]
)

class_counts_df

In [ ]:
# ==========================================================
# DISTRIBUIÇÃO DE CLASSES
# ==========================================================

plt.figure(figsize=(10, 5))

plt.bar(
    class_counts_df["Classe"],
    class_counts_df["Quantidade"]
)

plt.title("Distribuição de imagens por classe")
plt.xlabel("Classe")
plt.ylabel("Número de imagens")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# VISUALIZAÇÃO
# ==========================================================

plt.figure(figsize=(15, 10))

for i, class_name in enumerate(class_names):
    class_path = os.path.join(
        DATASET_PATH,
        class_name
    )

    files = [
        file for file in os.listdir(class_path)
        if file.lower().endswith(image_extensions)
    ]

    if len(files) == 0:
        continue

    image_path = os.path.join(
        class_path,
        files[0]
    )

    image = tf.keras.utils.load_img(image_path)

    ax = plt.subplot(
        2,
        int(np.ceil(NUM_CLASSES / 2)),
        i + 1
    )

    plt.imshow(image)
    plt.title(class_name)
    plt.axis("off")

plt.tight_layout()
plt.show()

### Upload e Divisão do dataset

O conjunto de dados será dividido em três partes:

- **70% para treinamento**
- **15% para validação**
- **15% para teste**

O conjunto de treinamento será utilizado para ajustar os pesos do modelo, já o conjunto de validação será utilizado durante o treinamento para acompanhar a capacidade de generalização do modelo para imagens que não foram utilizadas diretamente no ajuste dos pesos. O teste, por sua vez, será mantido separado durante o treinamento, sendo utilizado somente ao final para estimar o desempenho do modelo em imagens não utilizadas durante seu ajuste.

A separação será realizada em duas etapas:

1. 70% treinamento e 30% temporário;
2. O conjunto temporário será dividido igualmente em validação e teste.

*Observação*: por causa do arredondamento por batches, a proporção real pode não ser exatamente 70/15/15. Isso é normal e podemos registrar a quantidade efetiva de imagens depois.

In [ ]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.30,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

temporary_dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.30,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
temporary_batches = tf.data.experimental.cardinality(temporary_dataset).numpy()

print("Número de batches temporários:",temporary_batches)

validation_batches = temporary_batches // 2
validation_dataset = temporary_dataset.take(validation_batches)
test_dataset = temporary_dataset.skip(validation_batches)

print("Batches de validação:", tf.data.experimental.cardinality(validation_dataset).numpy())

print("Batches de teste:",tf.data.experimental.cardinality(test_dataset).numpy())

In [ ]:
class_names = train_dataset.class_names
NUM_CLASSES = len(class_names)

print("Classes utilizadas pelo TensorFlow:")

for index, name in enumerate(class_names):
    print(index, "->", name)

print("\nNúmero de classes:", NUM_CLASSES)

### Pré-processamento

In [ ]:
def preprocess_dataset(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

In [ ]:
# Treino
train_dataset = train_dataset.map(
    preprocess_dataset,
    num_parallel_calls=tf.data.AUTOTUNE
)

# Validação
validation_dataset = validation_dataset.map(
    preprocess_dataset,
    num_parallel_calls=tf.data.AUTOTUNE
)

# Teste
test_dataset = test_dataset.map(
    preprocess_dataset,
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

### Data Augmentation

Em problemas de classificação de imagens, o modelo pode apresentar dificuldade para generalizar quando o conjunto de treinamento não possui variedade suficiente. Para reduzir esse problema, será utilizado ***Data Augmentation***.

Durante o treinamento, algumas transformações aleatórias poderão ser
aplicadas às imagens, como:
- espelhamento horizontal;
- pequenas rotações;
- zoom;
- alteração de contraste.

Essas transformações não criam novas imagens armazenadas no disco: elas são aplicadas durante o treinamento, aumentando a diversidade das amostras apresentadas à rede. O objetivo é reduzir o risco de overfitting e tornar o modelo mais robusto a diferentes posições e condições de captura dos materiais.

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10)
    ],
    name="data_augmentation"
)

In [ ]:
for images, labels in train_dataset.take(1):
    sample_image = images[0]
    plt.figure(figsize=(12, 8))

    for i in range(6):
        augmented_image = data_augmentation(tf.expand_dims(sample_image, 0),training=True)[0]
        ax = plt.subplot(2, 3, i + 1)
        plt.imshow(tf.cast(tf.clip_by_value(augmented_image,0,255),tf.uint8))
        plt.axis("off")
    plt.suptitle("Exemplo de Data Augmentation", fontsize=18)
    plt.tight_layout()
    plt.show()
    break

------------
## ResNet-50

Será utilizada a arquitetura **ResNet-50**, originalmente proposta como uma rede residual profunda e amplamente utilizada em tarefas de classificação de imagens.

Neste projeto será utilizada uma versão pré-treinada no **ImageNet**.

A camada de classificação original da ResNet-50 será removida e uma nova cabeça de classificação será adicionada para as classes do *TrashNet*.

A arquitetura utilizada será:

```text
    Imagem 224 × 224 × 3
            │
            ▼
    Data Augmentation
            │
            ▼
    ResNet-50 pré-treinada 
            │
            ▼
    Global Average Pooling
            │
            ▼
        Dropout
            │
            ▼
    Dense + Softmax
            │
            ▼
    Classes do WasteNet

In [ ]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

print("ResNet-50 carregada.")
print("Camadas totais:", len(base_model.layers))

In [ ]:
inputs = tf.keras.Input(shape=(224, 224, 3),name="input_image")

x = data_augmentation(inputs)

x = base_model(x,training=False)

x = GlobalAveragePooling2D(name="global_average_pooling")(x)

x = Dropout(DROPOUT_RATE,name="dropout")(x)

outputs = Dense(NUM_CLASSES,activation="softmax",name="classifier")(x)

model = Model(inputs=inputs,outputs=outputs,name="ResNet50_WasteNet")

model.summary()

O modelo será compilado utilizando:

- **Otimizador:** Adam;
- **Learning rate:** `1 × 10⁻⁴`;
- **Função de perda:** Sparse Categorical Crossentropy;
- **Métrica:** Accuracy.

A função de perda `sparse_categorical_crossentropy` é adequada porque
as classes são representadas por números inteiros, por exemplo:

```text
    0 → cardboard
    1 → glass
    2 → metal
...

In [ ]:
model.compile(
    optimizer=Adam(
        learning_rate=LEARNING_RATE
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Callbacks

Durante o treinamento serão utilizados três mecanismos de controle:

- ***Early Stopping***:  Interrompe o treinamento quando o desempenho de validação deixa de melhorar por várias épocas, reduzindo o risco de overfitting e evita treinamento desnecessário.

- ***Model Checkpoint***: Salva automaticamente o modelo que apresentar o melhor desempenho na validação.

- ***Reduce Learning Rate***: Reduz a taxa de aprendizado quando a perda de validação deixa de melhorar.

In [ ]:
BEST_MODEL_PATH = os.path.join(
    MODELS_DIR,
    "resnet50_waste_best.keras"
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

### Transfer Learning

Nesta primeira fase, a ResNet-50 será utilizada como extratora de características. Os pesos aprendidos anteriormente no ImageNet serão mantidos congelados e somente a nova camada de classificação será treinada.

Essa estratégia é conhecida como ***Transfer Learning***, cujo objetivo é aproveitar as características visuais aprendidas pela ResNet-50, adaptando apenas a parte final da rede ao problema de classificação dos materiais recicláveis.

In [ ]:
history_initial = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_INITIAL,
    callbacks=[
        early_stopping,
        checkpoint,
        reduce_lr
    ]
)

### Fine-Tuning

Após o treinamento inicial, será realizada uma segunda etapa de adaptação chamada ***Fine-Tuning***. Nessa etapa, parte das camadas mais profundas da ResNet-50 será descongelada.

A taxa de aprendizado será reduzida para `1 × 10⁻⁵`. O learning rate menor é importante porque os pesos já possuem informações úteis provenientes do treinamento no ImageNet. Alterações muito grandes poderiam destruir essas características aprendidas.

As primeiras camadas permanecerão congeladas, enquanto parte das camadas finais poderá se adaptar às características específicas dos materiais recicláveis.

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer,layers.BatchNormalization):
        layer.trainable = False

print("Fine-Tuning a partir da camada:",FINE_TUNE_AT)

### Recompilação

In [ ]:
model.compile(
    optimizer=Adam(
        learning_rate=FINE_TUNING_LEARNING_RATE
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_finetune = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS_FINE_TUNING,
    callbacks=[
        early_stopping,
        checkpoint,
        reduce_lr
    ]
)

### Histórico

In [ ]:
history = {}

for key in history_initial.history:
    history[key] = (history_initial.history[key]+history_finetune.history[key])

history_df = pd.DataFrame(history)
history_df.head()

### Curvas de Treinamento

As curvas de *accuracy* e *loss* serão analisadas para verificar a evolução do modelo durante as etapas de *Transfer Learning* e *Fine-tuning*. A comparação entre treinamento e validação também permite identificar possíveis sinais de overfitting.

In [ ]:
initial_epochs = len(history_initial.history["accuracy"])

# Accuracy
plt.figure(figsize=(10, 6))
plt.plot(history["accuracy"],label="Treinamento")
plt.plot(history["val_accuracy"],label="Validação")
plt.axvline(initial_epochs - 1,linestyle="--",label="Início do Fine-Tuning")
plt.title("Accuracy durante o treinamento")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(
    os.path.join(
        RESULTS_DIR,
        "accuracy.png"
    ),
    dpi=300
)
plt.show()

In [ ]:
# Loss
plt.figure(figsize=(10, 6))
plt.plot(history["loss"],label="Treinamento")
plt.plot(history["val_loss"],label="Validação")
plt.axvline(initial_epochs - 1,linestyle="--",label="Início do Fine-Tuning")
plt.title("Loss durante o treinamento")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(
    os.path.join(
        RESULTS_DIR,
        "loss.png"
    ),
    dpi=300
)
plt.show()

### Carregando o melhor modelo

O modelo salvo pelo `ModelCheckpoing` corresponde à configuração que apresentou o melhor desempenho na validação. Esse modelo será carregado antes da avaliação final para garantir que as métricas sejam calculadas utilizando a melhor versão encontrada durante o treinamento.

In [ ]:
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)
print("Melhor modelo carregado")

------------------
## Avaliação

A avaliação final será realizada exclusivamente no conjunto de teste, que não participou do ajuste dos pesos e não foi utilizado para selecionar o melhor modelo. Serão calculadas:
- Accuracy;
- Precision;
- Recall;
- F1-Score.

In [ ]:
test_loss, test_accuracy = best_model.evaluate(test_dataset,verbose=1)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

### Predições

In [ ]:
y_true = []
y_pred = []
y_prob = []

for images, labels in test_dataset:
    probabilities = best_model.predict(images,verbose=0)
    predictions = np.argmax(probabilities,axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(predictions)
    y_prob.extend(probabilities)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print("Imagens avaliadas:",len(y_true))

### Métricas

In [ ]:
accuracy = accuracy_score(y_true,y_pred)

precision_macro = precision_score(y_true,y_pred,average="macro",zero_division=0)

recall_macro = recall_score(y_true,y_pred,average="macro",zero_division=0)

f1_macro = f1_score(y_true,y_pred,average="macro",zero_division=0)

precision_weighted = precision_score(y_true,y_pred,average="weighted",zero_division=0)

recall_weighted = recall_score(y_true,y_pred,average="weighted", zero_division=0)

f1_weighted = f1_score(y_true,y_pred,average="weighted",zero_division=0)

metrics_df = pd.DataFrame(
    {
        "Métrica": [
            "Accuracy",
            "Precision Macro",
            "Recall Macro",
            "F1-score Macro",
            "Precision Weighted",
            "Recall Weighted",
            "F1-score Weighted"
        ],

        "Valor": [
            accuracy,
            precision_macro,
            recall_macro,
            f1_macro,
            precision_weighted,
            recall_weighted,
            f1_weighted
        ]
    }
)

metrics_df

### Classification Repot

O *Classification Report* apresenta as métricas individualmente para cada classe. Essa análise é importante porqur uma Accuracy elevada, por exemplo, pode esconder um desempenho ruim em classes específicas.

In [ ]:
report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4,
    zero_division=0
)

print(report)

### Matriz de Confusão

A matriz de confusão apresenta a relação entre as classes reais e as classes previstas:
- Os elementos da diagonal principal representam classificações corretas.
- Os valores fora da diagonal representam erros de classificação.

Essa análise permite identificar quais tipos de materiais apresentam maior dificuldade para o modelo.

In [ ]:
cm = confusion_matrix(y_true,y_pred)

fig, ax = plt.subplots(figsize=(9, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=class_names)
disp.plot(ax=ax,xticks_rotation=45,colorbar=False)
plt.title("Matriz de Confusão — ResNet-50")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,"confusion_matrix.png"),dpi=300)
plt.show()

### Análise visual dos erros

Nesta etapa serão apresentadas algumas imagens do conjunto de teste com:
- classe real;
- classe prevista;
- confiança da previsão.

Imagens classificadas corretamente serão diferenciadas visualmente das imagens classificadas incorretamente. Essa análise qualitativa complementa as métricas quantitativas.

In [ ]:
plt.figure(figsize=(16, 12))

for images, labels in test_dataset.take(1):
    probabilities = best_model.predict(images,verbose=0)

    predictions = np.argmax(probabilities,axis=1)

    for i in range(min(12, len(images))):
        ax = plt.subplot(3,4,i + 1)
        image = images[i].numpy()

        # Normalização apenas para visualização
        image = image - image.min()

        if image.max() != 0:
            image = image / image.max()

        plt.imshow(image)

        true_class = class_names[labels[i].numpy()]
        predicted_class = class_names[predictions[i]]
        confidence = (probabilities[i][predictions[i]] * 100)

        if true_class == predicted_class:
            result = "CORRETO"
        else:
            result = "ERRO"

        plt.title(
            f"Real: {true_class}\n"
            f"Pred.: {predicted_class}\n"
            f"Conf.: {confidence:.1f}%\n"
            f"{result}"
        )
        plt.axis("off")

plt.tight_layout()
plt.show()

------------------
## Salvando o modelo

O modelo final será salvo no formato `.keras` e também será salvo um arquivo JSON contendo os nomes das classes na ordem utilziada pelo modelo. Esses dois arquivos serão utilizados no segundo notebook, que não precisará do *dataset* completo para executar a classificação em tepo real.

In [ ]:
# Melhor modelo salvo
FINAL_MODEL_PATH = os.path.join(MODELS_DIR,"resnet50_waste.keras")
best_model.save(FINAL_MODEL_PATH)
print(F"Modelo salvo em: {FINAL_MODEL_PATH}")

In [ ]:
CLASS_NAMES_PATH = os.path.join(MODELS_DIR,"class_names.json")

with open(CLASS_NAMES_PATH,"w",encoding="utf-8") as file:
    json.dump(class_names,file,ensure_ascii=False,indent=4)

print(f"Classes salvas em: {CLASS_NAMES_PATH}")

In [ ]:
# Métricas salvas
CLASS_NAMES_PATH = os.path.join(MODELS_DIR,"class_names.json")

with open(CLASS_NAMES_PATH,"w",encoding="utf-8") as file:
    json.dump(class_names,file,ensure_ascii=False,indent=4)

print(f"Classes salvas em: {CLASS_NAMES_PATH}")

In [ ]:
# Verifica se todos os arquivos foram gerados corretamente
print("=" * 60)
print("ARQUIVOS DO MODELO")
print("=" * 60)

for filename in os.listdir(MODELS_DIR):
    print(os.path.join(MODELS_DIR,filename))

print("\n")

print("=" * 60)
print("ARQUIVOS DE RESULTADOS")
print("=" * 60)

for filename in os.listdir(RESULTS_DIR):
    print(os.path.join(RESULTS_DIR,filename))

---------
## Resumo

O treinamento da ResNet-50 foi concluído.

O modelo foi desenvolvido utilizando Transfer Learning e Fine-Tuning, com imagens organizadas em classes de materiais recicláveis.

O experimento contemplou:
- análise exploratória do dataset;
- divisão dos dados em treinamento, validação e teste;
- pré-processamento das imagens;
- Data Augmentation;
- ResNet-50 pré-treinada no ImageNet;
- Transfer Learning;
- Fine-Tuning;
- Early Stopping;
- Model Checkpoint;
- redução adaptativa da taxa de aprendizado;
- avaliação no conjunto de teste;
- Accuracy;
- Precision;
- Recall;
- F1-score;
- matriz de confusão;
- análise visual das classificações.

**Arquivos principais**: 
O modelo e as classes foram armazenados em:
```text
models/
├── resnet50_waste.keras
└── class_names.json

Os resultados foram armazenados em:
results/
├── accuracy.png
├── loss.png
├── confusion_matrix.png
├── classification_report.txt
├── metrics.csv
└── training_history.csv